## **Introduction to Data Ingestion**

In [1]:
# ==========================================
# 1. Environment & Utility Imports
# ==========================================
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 2. Logging Setup
# ==========================================
import logging

logging.basicConfig(
    level=logging.INFO, 
    filename='app.log', 
    format='%(asctime)s - %(levelname)s - %(message)s', 
    filemode='w', 
    force=True  # Overrides any existing framework-configured root loggers
)

# ==========================================
# 3. LangChain Document & Splitter Imports
# ==========================================
# Updated LangChain Splitter Imports
from langchain_core.documents import Document

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

# Verify execution flow
logging.info("All Libraries Imported Successfully")


## **Document Structure In Langchain**

In [2]:
# Simple Document Creation
doc = Document(page_content="This is a sample document for testing.", 
               metadata={"source": "test_source",
                         "author": "chandan chaudhari",
                         "date": "2024-06-01",
                         "Infromation":" This is Test Document for Testing Purpose Only"})

logging.info(f"Document Created: {doc}")
print(f'content :{doc.page_content}')
print(f'metadata :{doc.metadata}')



content :This is a sample document for testing.
metadata :{'source': 'test_source', 'author': 'chandan chaudhari', 'date': '2024-06-01', 'Infromation': ' This is Test Document for Testing Purpose Only'}


In [3]:
import os

# 1. Define folder layout and ensure it exists
dir_path = "data/text_files"
os.makedirs(dir_path, exist_ok=True)

# 2. Content for both sample files
sample_files = {
    f"{dir_path}/sample_text1.txt": (
        "This is the first sample text file, designed specifically for testing text splitting functionality.\n"
        "It contains multiple lines of continuous paragraphs to evaluate document chunking performance.\n"
        "We will use different LangChain components like RecursiveCharacterTextSplitter to handle this content.\n"
        "The goal is to ensure that chunk limits are maintained without breaking key semantic sentences.\n"
        "By analyzing the outputs, we can optimize parameters like chunk size and chunk overlap effectively."
    ),
    f"{dir_path}/sample_text2.txt": (
        "This is the second sample text file, focusing on data extraction and metadata parsing pipelines.\n"
        "It simulates a secondary document structure containing technical workflow parameters.\n"
        "Once loaded, this file will be parsed into vector representations using dense embedding models.\n"
        "Managing multiple files helps verify that document loaders capture unique file paths properly.\n"
        "This structure forms the complete baseline for testing multi-document Retrieval-Augmented Generation."
    )
}

# 3. Physically write files to disk
for path, content in sample_files.items():
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Successfully generated: {path}")


Successfully generated: data/text_files/sample_text1.txt
Successfully generated: data/text_files/sample_text2.txt


In [4]:
file_path = "data/text_files/sample_text1.txt"

from langchain_community.document_loaders import TextLoader

# Read the file and create a Document object
loader = TextLoader(file_path, encoding="utf-8")

# Call the load method to extract the Document objects
docs = loader.load()

# Access the generated Document object
document = docs[0]

print(f"Document Content: {document.page_content}")
print(f"Document Metadata: {document.metadata}")



Document Content: This is the first sample text file, designed specifically for testing text splitting functionality.
It contains multiple lines of continuous paragraphs to evaluate document chunking performance.
We will use different LangChain components like RecursiveCharacterTextSplitter to handle this content.
The goal is to ensure that chunk limits are maintained without breaking key semantic sentences.
By analyzing the outputs, we can optimize parameters like chunk size and chunk overlap effectively.
Document Metadata: {'source': 'data/text_files/sample_text1.txt'}


In [5]:
# Load mutiple text files from the directory
from langchain_community.document_loaders import DirectoryLoader

dir_loader= DirectoryLoader("data/text_files", 
                                  glob="*.txt", 
                                  loader_cls=TextLoader, 
                                  loader_kwargs={"encoding": "utf-8"})

document = dir_loader.load()

for i,doc in enumerate(document):
    print(f"Document {i+1} Content: {doc.page_content}")
    print(f"Document {i+1} Metadata: {doc.metadata}")
    

Document 1 Content: This is the first sample text file, designed specifically for testing text splitting functionality.
It contains multiple lines of continuous paragraphs to evaluate document chunking performance.
We will use different LangChain components like RecursiveCharacterTextSplitter to handle this content.
The goal is to ensure that chunk limits are maintained without breaking key semantic sentences.
By analyzing the outputs, we can optimize parameters like chunk size and chunk overlap effectively.
Document 1 Metadata: {'source': 'data\\text_files\\sample_text1.txt'}
Document 2 Content: This is the second sample text file, focusing on data extraction and metadata parsing pipelines.
It simulates a secondary document structure containing technical workflow parameters.
Once loaded, this file will be parsed into vector representations using dense embedding models.
Managing multiple files helps verify that document loaders capture unique file paths properly.
This structure forms t

## **Text Split Techniques**

In [6]:
print(f"Total Documents Loaded: {len(document)}")
print(document[0].page_content)

Total Documents Loaded: 2
This is the first sample text file, designed specifically for testing text splitting functionality.
It contains multiple lines of continuous paragraphs to evaluate document chunking performance.
We will use different LangChain components like RecursiveCharacterTextSplitter to handle this content.
The goal is to ensure that chunk limits are maintained without breaking key semantic sentences.
By analyzing the outputs, we can optimize parameters like chunk size and chunk overlap effectively.


In [7]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(
    separator='\n', # New line character as the separator
    chunk_size=200,  # Maximum number of characters per chunk
    chunk_overlap=20, # Number of characters to overlap between chunks
    length_function=len # Function to calculate the length of each chunk (using Python's built-in len function)
)

chunk_char = splitter.split_documents(document)

chunk_char[0]

Document(metadata={'source': 'data\\text_files\\sample_text1.txt'}, page_content='This is the first sample text file, designed specifically for testing text splitting functionality.\nIt contains multiple lines of continuous paragraphs to evaluate document chunking performance.')

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter1 = RecursiveCharacterTextSplitter(
    chunk_size=200,       # Maximum target characters per chunk
    chunk_overlap=20,     # Overlapping characters between chunks
    length_function=len  # Calculate length using Python len()
)

chunk_recursive = splitter1.split_documents(document)

chunk_recursive[0]

Document(metadata={'source': 'data\\text_files\\sample_text1.txt'}, page_content='This is the first sample text file, designed specifically for testing text splitting functionality.\nIt contains multiple lines of continuous paragraphs to evaluate document chunking performance.')

In [9]:
from langchain_text_splitters import TokenTextSplitter

splitter2 = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=200,       # Maximum target tokens per chunk
    chunk_overlap=20      # Overlapping tokens between chunks
)

chunk_token = splitter2.split_documents(document)

chunk_token[0]

Document(metadata={'source': 'data\\text_files\\sample_text1.txt'}, page_content='This is the first sample text file, designed specifically for testing text splitting functionality.\nIt contains multiple lines of continuous paragraphs to evaluate document chunking performance.\nWe will use different LangChain components like RecursiveCharacterTextSplitter to handle this content.\nThe goal is to ensure that chunk limits are maintained without breaking key semantic sentences.\nBy analyzing the outputs, we can optimize parameters like chunk size and chunk overlap effectively.')

### Comparison of Three Text Splitters in LangChain

| Comparison Parameter | CharacterTextSplitter | RecursiveCharacterTextSplitter | TokenTextSplitter |
|---|---|---|---|
| Splitting strategy | Splits using a specified separator | Recursively splits using multiple separators | Splits based on tokens |
| Splitting unit | Characters | Characters | Tokens |
| Chunk size | Measured in characters | Measured in characters | Measured in tokens |
| Chunk overlap | Character-based | Character-based | Token-based |
| Default separator | `\n\n` | `["\n\n", "\n", " ", ""]` | Token boundaries |
| Sentence preservation | Not guaranteed | Better chance of preserving sentences | Not guaranteed |
| Paragraph preservation | Depends on separator | Attempts to preserve paragraphs | Not guaranteed |
| Context preservation | Basic | Better structural preservation | Depends on token boundaries |
| Token awareness | No | No | Yes |
| Chunk size control | Target size; can exceed it | Better at respecting target size | Token-based control |
| Best suited for | Simple text documents | General-purpose RAG applications | LLM token-budget management |
| Typical application | Basic text processing | PDF, DOCX and knowledge-base RAG | Token-constrained LLM pipelines |
| Recommendation | Basic | Recommended starting point | Useful for model-specific requirements |

In [ ]:
# Load pdf Document
from langchain_community.document_loaders import PyPDFLoader, UnstructuredPDFLoader,PyMuPDFLoader, PDFPlumberLoader

try:
    pdf_loader = PyPDFLoader("data/pdf_files/Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.pdf")
    pdf_document = pdf_loader.load()
    print(f"Total PDF Documents Loaded: {len(pdf_document)}")
    print(pdf_document[0].page_content)
except Exception as e:
    print(f"Error loading PDF document: {e}")

Total PDF Documents Loaded: 19
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University College London;⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-
stream NLP tasks. However, their ability to access and precisely manipulate knowl-
edge is still limited, and hence on knowledge-intensive tasks, their performance
lags behind task-speciﬁc architectures. Additionally, providing provenance for their
decisions and updating their world knowledge remain open research problems. Pre-
trained models with a differentiable access mechanism to explicit non-parametric
memory have so far been only investigate

In [18]:
# Load pdf Document using PyMuPDFLoader
try:
    pymupdf_loader = PyMuPDFLoader("data/pdf_files/Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.pdf")
    pymupdf_document = pymupdf_loader.load()
    print(f"Total PyMuPDF Documents Loaded: {len(pymupdf_document)}")
    print(pymupdf_document[0].page_content)
except Exception as e:
    print(f"Error loading Unstructured PDF document: {e}")

Total PyMuPDF Documents Loaded: 19
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research; ‡University College London; ⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-
stream NLP tasks. However, their ability to access and precisely manipulate knowl-
edge is still limited, and hence on knowledge-intensive tasks, their performance
lags behind task-speciﬁc architectures. Additionally, providing provenance for their
decisions and updating their world knowledge remain open research problems. Pre-
trained models with a differentiable access mechanism to explicit non-parametric
memory have so far been only inves